In [ ]:
import os
os.environ['GROQ_API_KEY'] = '...'

In [88]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled, NoTranscriptFound
from langchain_text_splitters import RecursiveCharacterTextSplitter,Language
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

In [89]:
# Fetching the transcript of a YouTube video
# step1 : Data Ingestion
video_id = "LPZh9BOjkQs"
try:
    ytt_api = YouTubeTranscriptApi()
    transcript_list = ytt_api.fetch(video_id, languages=['en'])

    transcript = " ".join(chunk.text for chunk in transcript_list)
    print(transcript)

except TranscriptsDisabled:
    print("Transcripts are disabled for this video.")

Imagine you happen across a short movie script that describes a scene between a person and their AI assistant. The script has what the person asks the AI, but the AI's response has been torn off. Suppose you also have this powerful magical machine that can take any text and provide a sensible prediction of what word comes next. You could then finish the script by feeding in what you have to the machine, seeing what it would predict to start the AI's answer, and then repeating this over and over with a growing script completing the dialogue. When you interact with a chatbot, this is exactly what's happening. A large language model is a sophisticated mathematical function that predicts what word comes next for any piece of text. Instead of predicting one word with certainty, though, what it does is assign a probability to all possible next words. To build a chatbot, you lay out some text that describes an interaction between a user and a hypothetical AI assistant, add on whatever the use

In [90]:
# step2 : Indexing(Text Splitting and chunking)
splitter=RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks=splitter.create_documents([transcript])

In [91]:
# step3 : Indexing(Embedding generation and storing in vectorstore)
embeddings=HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore=FAISS.from_documents(chunks, embeddings)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3192.18it/s]


In [92]:
# step3 : Retrieval
retriever=vectorstore.as_retriever(search_type='similarity', search_kwargs={"k":4})

In [93]:
retriever.invoke('What is deepmind?')

[Document(id='98522b8c-07d5-4c1a-b159-e7722c8549e6', metadata={}, page_content="Imagine you happen across a short movie script that describes a scene between a person and their AI assistant. The script has what the person asks the AI, but the AI's response has been torn off. Suppose you also have this powerful magical machine that can take any text and provide a sensible prediction of what word comes next. You could then finish the script by feeding in what you have to the machine, seeing what it would predict to start the AI's answer, and then repeating this over and over with a growing script completing the dialogue. When you interact with a chatbot, this is exactly what's happening. A large language model is a sophisticated mathematical function that predicts what word comes next for any piece of text. Instead of predicting one word with certainty, though, what it does is assign a probability to all possible next words. To build a chatbot, you lay out some text that describes an int

In [94]:
# step4 : Augmentation
llm=ChatGroq(model="llama-3.1-8b-instant", temperature=0.9)

prompt=PromptTemplate(
    template="""You are a helpful assistant. \
    Answer only from the provided transcript context \
    If the context is insufficient,just say you dont know.\
    
    {context} \
    Question:{question}""",
    input_variables=["context","question"]
)

In [95]:
question='is the topic of llm discussed in this video?if yes then what was discussed'
retrieved_docs=retriever.invoke(question)

In [96]:
context_text="\n\n".join(doc.page_content for doc in retrieved_docs)

In [97]:
final_prompt=prompt.invoke({"context": context_text, "question": question})

In [98]:
# step4 : Generation
answer=llm.invoke(final_prompt)
print(answer.content)

Yes, the topic of LLM (Large Language Model) is discussed in this video. 

The video discusses the basics of how Large Language Models work, specifically mentioning that they are sophisticated mathematical functions that predict what word comes next for any piece of text. It also explains that these models use two fundamental operations: self-attention and position-wise feed-forward neural network.


In [99]:
############################################################################

In [100]:
# Building a Chain
from langchain_core.runnables import RunnableParallel,RunnablePassthrough,RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [101]:
def format_docs(retrieved_docs):
    context_text="\n\n".join(doc.page_content for doc in retrieved_docs)
    return context_text

In [102]:
parallel_chain=RunnableParallel(
    {'context':retriever | RunnableLambda(format_docs),
     'question':RunnablePassthrough()
    })

In [103]:
parallel_chain.invoke('what is rag')

{'context': "called reinforcement learning with human feedback. Workers flag unhelpful or problematic predictions, and their corrections further change the model's parameters, making them more likely to give predictions that users prefer. Looking back at the pre-training, though, this staggering amount of computation is only made possible by using special computer chips that are optimized for running many operations in parallel, known as GPUs. However, not all language models can be easily parallelized. Prior to 2017, most language models would process text one word at a time, but then a team of researchers at Google introduced a new model known as the transformer. Transformers don't read text from the start to the finish, they soak it all in at once, in parallel. The very first step inside a transformer, and most other language models for that matter, is to associate each word with a long list of numbers. The reason for this is that the training process only works with continuous valu

In [104]:
parser=StrOutputParser()

In [105]:
main_chain=parallel_chain | prompt | llm | parser

In [106]:
main_chain.invoke('Can you summarize the video')

"The video discusses how a large language model, specifically a transformer, works. It explains that the model consists of two fundamental operations: self-attention and a feed-forward neural network. These operations are repeated multiple times with the model's prediction improving with each iteration. \n\nThe model takes in a sequence of numbers, which are then processed and enriched to encode information about the input text and trained patterns. The final vector is used to produce a prediction of the next word, which is a probability for every possible next word.\n\nThis concept is shown through an example of a movie script where the AI's response is torn off, and a magical machine is used to predict the next word. The video explains that when interacting with a chatbot, this is exactly what's happening with a large language model predicting the next word with a probability for all possible options."